# Notebook 07: Dataset Table, Skill Database, and Dataset Catalog

This notebook generates a unified table from the contents of all external datasets for profile expansion, as well as an initial skill database/vocabulary and a dataset catalog of all datasets for documentation purposes in the thesis.

## 1. Dataset Table: documents_raw.parquet

All external text datasets in the Unified Document Schema: job ads (2 datasets), BA jobs, Workwise scraping, CV dataset, LinkedIn profiles, Udemy courses.

Purpose: A central table accessed by all subsequent methods. Central document through which the base profiles are extended.
Included files from data/interim_external/ among others:
- ba_job_ads_unified.parquet
- techsalerator_job_postings_unified.parquet
- postings_big_unified.parquet
- resume_structured_unified.parquet
- linkedin_morocco_unified.parquet
- courses_udemy_unified.parquet
- scraped_workwise_dynamic_unified.parquet
- Not included: scraped_workwise_unified.parquet (from Scrap Method A as a demo) and ba_* Parquets (standards, used as a skills database, no free text)

In [1]:
# Imports + Paths
from pathlib import Path
import pandas as pd

# Project Root
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_INTERIM_EXTERNAL = PROJECT_ROOT / "data" / "interim_external"
DATA_PROCESSED_EXTERNAL = PROJECT_ROOT / "data" / "processed_external"

for p in [DATA_INTERIM_EXTERNAL, DATA_PROCESSED_EXTERNAL]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_INTERIM_EXTERNAL =", DATA_INTERIM_EXTERNAL)
print("DATA_PROCESSED_EXTERNAL =", DATA_PROCESSED_EXTERNAL)

PROJECT_ROOT = C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
DATA_INTERIM_EXTERNAL = C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external
DATA_PROCESSED_EXTERNAL = C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external


In [2]:
unified_files = {
    "kaggle_linkedin_big": "postings_big_unified.parquet",
    "techsalerator": "techsalerator_job_postings_unified.parquet",
    "resume_structured": "resume_structured_unified.parquet",
    "linkedin_morocco": "linkedin_morocco_unified.parquet",
    "udemy_courses": "courses_udemy_unified.parquet",
    "workwise_selenium": "scraped_workwise_dynamic_unified.parquet",
    "ba_job_ads": "ba_job_ads_unified.parquet",
}

Import & check per record:

In [3]:
dfs = []

for name, fname in unified_files.items():
    path = DATA_INTERIM_EXTERNAL / fname
    print(f"\nLese {name}: {path}")
    df = pd.read_parquet(path)
    print(f"{name}: {len(df)} Zeilen")
    print(df.head(2))
    print(df.dtypes)
    dfs.append(df)


Lese kaggle_linkedin_big: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\postings_big_unified.parquet
kaggle_linkedin_big: 123849 Zeilen
    doc_id source_type                    source_name  \
0   921716      job_ad  kaggle_linkedin_2023_2024_big   
1  1829192      job_ad  kaggle_linkedin_2023_2024_big   

                       job_title_raw  \
0              Marketing Coordinator   
1  Mental Health Therapist/Counselor   

                                            raw_text language  \
0  Marketing Coordinator\n\nJob descriptionA lead...       en   
1  Mental Health Therapist/Counselor\n\nAt Aspen ...       en   

                                           meta_json  
0  {"job_id": 921716, "company_id": 2774458.0, "l...  
1  {"job_id": 1829192, "company_id": NaN, "locati...  
doc_id           object
source_type      object
source_name      object
job_title_raw    object
raw_text         object
language         object
meta_json    

All records that have already been transformed into the Unified Document Schema are read from `data/interim_external/`. For each record, the following are briefly checked:
- Path and filename (`unified_files`)
- Table header (`head(2)`)
- Column names and data types

All sources are already in the same schema  (`doc_id, source_type, source_name, job_title_raw, raw_text, language, meta_json`) and can now be merged.

Merge all DataFrames: Check the number of rows and the distribution by source_type/source_name

In [4]:
df_docs = pd.concat(dfs, ignore_index=True)

df_docs.info()
df_docs[["source_type", "source_name"]].value_counts().head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292109 entries, 0 to 292108
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   doc_id         292109 non-null  object
 1   source_type    292109 non-null  object
 2   source_name    292109 non-null  object
 3   job_title_raw  292109 non-null  object
 4   raw_text       292109 non-null  object
 5   language       194005 non-null  object
 6   meta_json      292109 non-null  object
dtypes: object(7)
memory usage: 15.6+ MB


source_type       source_name                  
job_ad            kaggle_linkedin_2023_2024_big    123849
course            kaggle_udemy_courses              98104
cv                kaggle_resume_structured          54933
job_ad            kaggle_techsalerator               9807
                  ba_jobsuche_api_2025               3505
linkedin_profile  kaggle_linkedin_morocco            1525
job_ad_scrape     workwise_selenium                   386
Name: count, dtype: int64

All imported UnifiedDataFrames are combined using `pd.concat` into a single document table named `df_docs`. Result:
- ~292k documents (rows)
- Uniform schema with 7 columns: `doc_id, source_type, source_name, job_title_raw, raw_text, language, meta_json`
- Source distribution:
  - 2 job postings (medium 10k and large Kaggle dataset 124k)
  - BA API job postings 3505
  - LinkedIn profiles (Morocco) 1525
  - Resume dataset 55k
  - Udemy courses 98k
  - Workwise scraping (Selenium) 609

This table will later form the basis for all data-driven methods and is saved as `documents_raw.parquet`. Before that, check the formats:

In [5]:
df_docs["doc_id"].map(type).value_counts()

doc_id
<class 'str'>    194005
<class 'int'>     98104
Name: count, dtype: int64

Set the missing source_type for BA API job listings:

In [6]:
mask_ba = df_docs["source_name"] == "ba_jobsuche_api_2025"

# check beforehand
print("BA source_type before:")
print(df_docs.loc[mask_ba, "source_type"].value_counts(dropna=False))

# set
df_docs.loc[mask_ba, "source_type"] = "job_ad"

# check
print("BA source_type after:")
print(df_docs.loc[mask_ba, "source_type"].value_counts())

BA source_type before:
source_type
job_ad    3505
Name: count, dtype: int64
BA source_type after:
source_type
job_ad    3505
Name: count, dtype: int64


In some records, the data is stored as a string, while in others it is stored as an integer. Solution: Consistently convert doc_id to strings:

In [7]:
import json
import pandas as pd

# Always represent meta_json as a string; treat NaN or missing meta as {}
def normalize_meta(x):
    if pd.isna(x): # missing values -> empty string
        return "{}"
    # If dict/list already exists, dump it cleanly as JSON
    if isinstance(x, (dict, list)):
        return json.dumps(x, ensure_ascii=False)
    s = str(x).strip() # otherwise convert to a string
    return s if s else "{}"

df_docs["meta_json"] = df_docs["meta_json"].apply(normalize_meta)

doc_id should be unique:

In [8]:
dup = df_docs["doc_id"].duplicated().sum()
print("Duplicate doc_id:", dup)

Duplicate doc_id: 14


doc_id should serve as a unique key for future joins/lookups. Make the affected rows unique here by adding a fixed suffix to all occurrences except the first one:
doc_id = source_name + “::” + doc_id + “::dup<k>”

In [9]:
# Highlight duplicates
dup_mask = df_docs["doc_id"].duplicated(keep=False)

n_dup_rows = int(dup_mask.sum())
n_dup_groups = int(df_docs.loc[dup_mask, "doc_id"].nunique())

print(f"Duplicate groups (unique doc_id values): {n_dup_groups}")
print(f"Rows involved in duplicates: {n_dup_rows}")

if n_dup_rows > 0:
    # A consistent order within the doc_id group
    dup_rank = df_docs.loc[dup_mask].groupby("doc_id").cumcount()

    # Change ID
    fix_mask = dup_mask.copy()
    fix_mask.loc[dup_mask] = dup_rank >= 1

    df_docs.loc[fix_mask, "doc_id"] = (
        df_docs.loc[fix_mask, "source_name"].astype(str)
        + "::"
        + df_docs.loc[fix_mask, "doc_id"].astype(str)
        + "::dup"
        + (dup_rank[dup_rank >= 1] + 1).astype(str).values
    )

# Check
dup_after = int(df_docs["doc_id"].duplicated().sum())
print("Duplicate doc_id after fix:", dup_after)

Duplicate groups (unique doc_id values): 10
Rows involved in duplicates: 24
Duplicate doc_id after fix: 0


Harmonize language:

The `language` column is populated consistently for all documents so that subsequent steps (cleaning, tokenization, feature extraction) can operate based on language.

Procedure (2-step):
1. Populate using `source_name` if the language is certain (e.g., Workwise = `de`, Udemy = `en`).
2. Automatic language detection only for mixed sources or remaining `None`/empty values:
   - `kaggle_techsalerator` (mixed `de/en`)
   - `kaggle_linkedin_morocco` (mixed `en/fr/ar`)
   - only on the remaining subset, for performance reasons

In [10]:
import numpy as np

# Clear the language column
def _norm_lang(x):
    if x is None:
        return np.nan
    if isinstance(x, float) and np.isnan(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"", "none", "nan"}:
        return np.nan
    return s

df_docs["language"] = df_docs["language"].apply(_norm_lang)

# Safe mapping based on source_name if empty
LANG_BY_SOURCE_SURE = {
    "workwise_selenium": "de",
    "ba_jobsuche_api_2025": "de",
    "kaggle_linkedin_2023_2024_big": "en",
    "kaggle_resume_structured": "en",
    "kaggle_udemy_courses": "en",
}

for src, lang in LANG_BY_SOURCE_SURE.items():
    mask = (df_docs["source_name"] == src) & (df_docs["language"].isna())
    df_docs.loc[mask, "language"] = lang

# Quick-Check, missing?
missing_after_sure = df_docs["language"].isna().sum()
missing_after_sure

np.int64(0)

In [11]:
# Check the distribution by source_name to see if it matches expectations
df_docs.groupby("source_name")["language"].apply(lambda s: s.isna().mean()).sort_values(ascending=False).head(15)

source_name
ba_jobsuche_api_2025             0.0
kaggle_linkedin_2023_2024_big    0.0
kaggle_linkedin_morocco          0.0
kaggle_resume_structured         0.0
kaggle_techsalerator             0.0
kaggle_udemy_courses             0.0
workwise_selenium                0.0
Name: language, dtype: float64

Automatic language detection (subset only): `langdetect` is used for mixed sources and any remaining missing `language` values.

In [ ]:
# %pip install langdetect # required only once

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\sigle\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [16]:
# Test
from langdetect import detect
detect("Das ist Deutsch.")

'de'

In [17]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0 

MIXED_SOURCES = {
    "kaggle_techsalerator", # de/en/evtl. other languages mixed
    "kaggle_linkedin_morocco", # en/fr/ar mixed
}

# only Mixed Sources and only if language is missing
subset_mask = df_docs["source_name"].isin(MIXED_SOURCES) & (df_docs["language"].isna())

to_detect = df_docs.loc[subset_mask, ["raw_text"]].copy() # Check raw_text

def safe_detect_lang(text: str):
    if text is None:
        return np.nan
    t = str(text).strip()
    if len(t) < 50:
        return np.nan
    # an excerpt
    t = t[:1000]
    try:
        return detect(t)
    except Exception:
        return np.nan

df_docs.loc[subset_mask, "language"] = to_detect["raw_text"].apply(safe_detect_lang)
df_docs["language"].value_counts(dropna=False).head(20)

language
en       285475
de         5131
pt          535
es          274
fr          172
pl          115
cs           95
nl           87
sl           68
hu           47
ro           35
sk           18
tr           15
da           14
ja           12
ru            4
it            3
uk            3
zh-cn         2
vi            2
Name: count, dtype: int64

Language distribution of the datasets

Harmonize data types before saving:

In [18]:
# doc_id = string
df_docs["doc_id"] = df_docs["doc_id"].astype("string")

# remaining columns
for col in ["source_type", "source_name", "job_title_raw", "raw_text", "language", "meta_json"]:
    if col in df_docs.columns:
        df_docs[col] = df_docs[col].astype("string")

# missing values as <NA> instead of “nan”
df_docs["language"] = df_docs["language"].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
df_docs["meta_json"] = df_docs["meta_json"].replace({"nan": "{}", "None": "{}", "": "{}"})

# Quick check to make sure they're all strings
df_docs[["doc_id", "language", "meta_json"]].dtypes

doc_id       string[python]
language     string[python]
meta_json    string[python]
dtype: object

In [19]:
# Save as documents_raw.parquet
output_path = DATA_PROCESSED_EXTERNAL / "documents_raw.parquet"
df_docs.to_parquet(output_path, index=False, engine="pyarrow")
print("Gespeichert unter:", output_path)

Gespeichert unter: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\documents_raw.parquet


Sample output:

In [20]:
# Reload `documents_raw.parquet` and take a quick look at it
cols_light = ["doc_id", "source_type", "source_name", "job_title_raw", "language"]

df_docs_check = pd.read_parquet(
    output_path,
    engine="pyarrow",
    columns=cols_light
)

print("Anzahl Dokumente:", len(df_docs_check))
print(df_docs_check[["source_type", "source_name"]].value_counts().head(10))

# Examples (simple because raw_text/meta_json isn't loaded)
df_docs_check.sample(10, random_state=42)

Anzahl Dokumente: 292109
source_type       source_name                  
job_ad            kaggle_linkedin_2023_2024_big    123849
course            kaggle_udemy_courses              98104
cv                kaggle_resume_structured          54933
job_ad            kaggle_techsalerator               9807
                  ba_jobsuche_api_2025               3505
linkedin_profile  kaggle_linkedin_morocco            1525
job_ad_scrape     workwise_selenium                   386
Name: count, dtype: int64


,doc_id,source_type,source_name,job_title_raw,language
153674,resume_struct_20019,cv,kaggle_resume_structured,Freelance Corporate Trainer,en
275596,4261228,course,kaggle_udemy_courses,Guitar Essentials for Beginners,en
234391,3985196,course,kaggle_udemy_courses,Introduction to Song Tantra - Songwriting from...,en
270704,3473188,course,kaggle_udemy_courses,Build Strong Immune system through healing spices,en
274423,5067872,course,kaggle_udemy_courses,Ultimate Way Of Healing Anxiety,en
146990,resume_struct_13335,cv,kaggle_resume_structured,Sr. Full stack Java Developer,en
4244,3884899485,job_ad,kaggle_linkedin_2023_2024_big,LPN,en
219531,3572969,course,kaggle_udemy_courses,Dynamics 365 Finance&Operations: Financials Pa...,en
268360,4808674,course,kaggle_udemy_courses,Facial Diagnosis (Face Mapping),en
82993,3904054091,job_ad,kaggle_linkedin_2023_2024_big,Forklift Operator,en


Sample check, table output:

- the total number of documents
- the distribution by `source_type`/`source_name`
- random sample rows

## 2. Skill Database: skills_vocab_raw.csv

Objective: To create an initial skill vocabulary list from ESCO skills (data/interim/esco_skills.parquet) and the BA skill base (data/interim_external/ba_total.parquet) in a table named skills_vocab_raw.csv. This serves as a central skill vocabulary list/database as the basis for profile expansion methods. (Methodology based on Pinzone et al. (2024) and Malherbe et al. (2016))

In [21]:
# Imports
import re
import ast

Load Esco Skills:

In [22]:
esco_path = DATA_INTERIM / "esco_skills.parquet"
df_esco = pd.read_parquet(esco_path)

print("ESCO-Skills:", esco_path)
df_esco.head(3)
df_esco.dtypes

ESCO-Skills: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim\esco_skills.parquet


skill_uri         object
pref_label_en     object
alt_labels_en     object
description_en    object
reuse_level       object
skill_type        object
dtype: object

ESCO skills that have already been processed from `esco_skills.parquet` are loaded, along with the columns listed above.

Load the BA skill base (ba_total.parquet):

In [23]:
ba_path = DATA_INTERIM_EXTERNAL / "ba_total.parquet"
df_ba = pd.read_parquet(ba_path)

print("BA-Skillbasis:", ba_path)
df_ba.head(5)
df_ba.dtypes

BA-Skillbasis: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\ba_total.parquet


skill_id           object
code               object
canonical_label    object
synonym            object
synonym_tech       object
group_id           object
group_code         object
group_label        object
dtype: object

ba_total.parquet contains the Federal Agency's skill database (DKZ competencies + keywords). The table primarily provides German-language labels and synonyms. In the next step, the columns will be mapped to the common vocabulary format.

Utility function for parsing alt labels (for ESCO): safely converting alt_labels_en into a list. The split_alt_labels function normalizes these formats and always returns a clean list of synonyms.

In [24]:
def split_alt_labels(value): # Returns altLabels in various formats and as a list of strings, with NaN/None -> [], list -> as-is, JSON string -> ast.literal_eval, and regular string -> split at ; or | or ,
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    # already a list?
    if isinstance(value, list):
        return [s.strip() for s in value if isinstance(s, str) and s.strip()]

    # Then try to parse the list
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        # JSON-like; starts with [ or {
        if s[0] in "[{":
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(x).strip() for x in parsed if str(x).strip()]
            except (ValueError, SyntaxError):
                pass
        # Fallback: Split at typical splitters
        parts = re.split(r"[;|,]", s)
        return [p.strip() for p in parts if p.strip()]

    # everything else
    return []

ESCO Skills -> Entries in the skill vocabulary. Specific column names from df_esco:
- skill_uri -> Skill ID
- pref_label_en -> Preferred Label (English)
- alt_labels_en -> Alternative Labels (synonyms, as a string or list as needed)

In [25]:
vocab_rows = []

for _, row in df_esco.iterrows():
    skill_id = row["skill_uri"]

    # Preferred Label (english)
    pref_en = row.get("pref_label_en")
    if isinstance(pref_en, str) and pref_en.strip():
        vocab_rows.append({
            "vocab_id":      f"ESCO:{skill_id}:pref_en",
            "label":         pref_en.strip(),
            "source_system": "ESCO",
            "source_id":     skill_id,
            "lang":          "en",
            "label_type":    "preferred",
        })

    # AltLabels (english)
    alt_val = row.get("alt_labels_en")
    for alt in split_alt_labels(alt_val):
        vocab_rows.append({
            "vocab_id":      f"ESCO:{skill_id}:alt_en:{alt}",
            "label":         alt,
            "source_system": "ESCO",
            "source_id":     skill_id,
            "lang":          "en",
            "label_type":    "alt",
        })

BA Skill Base --> Entries in the skill vocabulary. Mapping of the BA columns:
- canonical_label forms the main label for each BA code (label_type = “ba_label”)
- synonym and synonym_tech as additional German-language variants (‘ba_synonym’, “ba_synonym_tech”)
- Thus, the vocabulary combines an English ESCO skill base with a German-language BA base that is rich in synonyms.

In [26]:
for _, row in df_ba.iterrows():
    code = row["code"]

    # Official BA designation
    canon = row.get("canonical_label")
    if isinstance(canon, str) and canon.strip():
        vocab_rows.append({
            "vocab_id":      f"BA:{code}:label",
            "label":         canon.strip(),
            "source_system": "BA",
            "source_id":     code,
            "lang":          "de",
            "label_type":    "ba_label",
        })

    # Synonyme
    syn = row.get("synonym")
    if isinstance(syn, str) and syn.strip():
        vocab_rows.append({
            "vocab_id":      f"BA:{code}:syn:{syn.strip()}",
            "label":         syn.strip(),
            "source_system": "BA",
            "source_id":     code,
            "lang":          "de",
            "label_type":    "ba_synonym",
        })

    # technische Synonyms
    syn_tech = row.get("synonym_tech")
    if isinstance(syn_tech, str) and syn_tech.strip():
        vocab_rows.append({
            "vocab_id":      f"BA:{code}:syntech:{syn_tech.strip()}",
            "label":         syn_tech.strip(),
            "source_system": "BA",
            "source_id":     code,
            "lang":          "de",
            "label_type":    "ba_synonym_tech",
        })

Create, clean, and save a Skill DB/Vocabulary DataFrame:

In [27]:
df_vocab = pd.DataFrame(vocab_rows)
print("Rohgröße Vokabular:", df_vocab.shape)

# Remove empty labels
df_vocab["label"] = df_vocab["label"].astype(str).str.strip()
df_vocab = df_vocab[df_vocab["label"] != ""].copy()

# Remove duplicate entries (same source + label)
before = len(df_vocab)
df_vocab = (
    df_vocab
    .drop_duplicates(subset=["source_system", "source_id", "label"])
    .reset_index(drop=True)
)
after = len(df_vocab)
print(f"Zeilen vor/nach Deduplikation: {before} -> {after}")

# Quick Check
df_vocab.info()
df_vocab.head(10)
df_vocab["source_system"].value_counts()
df_vocab["label_type"].value_counts()

Rohgröße Vokabular: (161908, 6)
Zeilen vor/nach Deduplikation: 161908 -> 111727
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111727 entries, 0 to 111726
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   vocab_id       111727 non-null  object
 1   label          111727 non-null  object
 2   source_system  111727 non-null  object
 3   source_id      111727 non-null  object
 4   lang           111727 non-null  object
 5   label_type     111727 non-null  object
dtypes: object(6)
memory usage: 5.1+ MB


label_type
ba_synonym_tech    48282
ba_synonym         39773
preferred          13939
ba_label            9733
Name: count, dtype: int64

- Raw size of the vocabulary: 161,898 rows; after deduplication: 111,727 rows -> 50,126 duplicates removed
- Column structure of the final table:
    - vocab_id = unique identifier for each skill variant
    - label = actual skill text
    - source_system = source (ESCO or BA)
    - source_id = skill URI (ESCO) or BA competency code
    - lang  = language (en/de)
    - label_type = Label type: preferred, alt, ba_label, ba_synonym, ba_synonym_tech
- Distribution by label type: BA contributes the majority of synonym variants; ESCO provides the structured preferred labels and English alt labels
    - ba_synonym_tech   48,282
    - ba_synonym    39,773
    - preferred (ESCO)  13,939
    - ba_label  9,733

Save to data/processed_external/skills_vocab_raw.csv:

In [28]:
output_vocab = DATA_PROCESSED_EXTERNAL / "skills_vocab_raw.csv"
df_vocab.to_csv(output_vocab, index=False, encoding="utf-8")

print("Skills-Vokabular gespeichert unter:", output_vocab)

Skills-Vokabular gespeichert unter: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_raw.csv


## 3. Dataset Catalog: dataset_catalog.csv

Centralized documentation of all datasets used in the thesis. A dataset catalog that documents all data sources used: datasets used, size, source, type (job_ad, cv, course, scrape, standard), and suitability/utility.

In [29]:
import pyarrow.parquet as pq

# Paths
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_INTERIM_EXTERNAL = PROJECT_ROOT / "data" / "interim_external"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED_EXTERNAL = PROJECT_ROOT / "data" / "processed_external"

DOCS_PATH = PROJECT_ROOT / "docs"
DOCS_PATH.mkdir(exist_ok=True)

Helper function for collecting metadata:

In [30]:
catalog_rows = []

def _count_rows(path: Path): # counts lines in a memory-efficient way
    if not path.exists():
        return None

    suf = path.suffix.lower()
    try:
        if suf == ".parquet":
            return pq.ParquetFile(path).metadata.num_rows # Metadata
        elif suf == ".csv":
            with path.open("r", encoding="utf-8", errors="ignore") as f: # Count lines without full parsing (exclude headers)
                return max(sum(1 for _ in f) - 1, 0)
        else:
            return None
    except Exception:
        return None

def add_dataset(id, dataset_name, dtype, path, source_name, notes=""):
    p = Path(path)

    n_rows = _count_rows(p)
    fmt = p.suffix.lstrip(".") if p.suffix else None

    catalog_rows.append({
        "id": id,
        "dataset_name": dataset_name,
        "type": dtype,
        "format": fmt,
        "path": str(p),
        "n_rows": n_rows,
        "source_name": source_name,
        "notes": notes,
    })

Add records to the catalog:

In [31]:
# A. Unified document sources (in data/interim_external/)
add_dataset(
    "D1", "Techsalerator Job Postings", "job_ad",
    DATA_INTERIM_EXTERNAL / "techsalerator_job_postings_unified.parquet",
    "kaggle_techsalerator",notes="≈9.8k englische Jobanzeigen; zusätzliche Jobmarkt-Quelle neben LinkedIn."
)

add_dataset(
    "D2", "LinkedIn Job Postings (big)", "job_ad",
    DATA_INTERIM_EXTERNAL / "postings_big_unified.parquet",
    "kaggle_linkedin_2023_2024_big", notes="≈123.8k Anzeigen; große Basis für robuste Co-Occurrence/Mining-Analysen."
)

add_dataset(
    "D3", "BA Job Ads (Jobsuche API 2025)", "job_ad",
    DATA_INTERIM_EXTERNAL / "ba_job_ads_unified.parquet",
    "ba_jobsuche_api_2025", notes="≈3.5k deutsche Jobanzeigen; eigene/operative Quelle (BA Jobsuche API)."
)

add_dataset(
    "D4", "Resume Dataset (Structured)", "cv",
    DATA_INTERIM_EXTERNAL / "resume_structured_unified.parquet",
    "kaggle_resume_structured", notes="≈54.9k CVs/Resumes; strukturierter Resume-Datensatz (Skill-Extraktion/Abgleich)."
)

add_dataset(
    "D5", "LinkedIn Morocco Profiles", "linkedin_profile",
    DATA_INTERIM_EXTERNAL / "linkedin_morocco_unified.parquet",
    "kaggle_linkedin_morocco", notes="≈1.5k Profile; Social/Profil-Quelle für Co-Occurrence/Skills im Profiltext."
)

add_dataset(
    "D6", "Udemy Courses", "course",
    DATA_INTERIM_EXTERNAL / "courses_udemy_unified.parquet",
    "kaggle_udemy_courses", notes="≈98.1k Kurse; Titel sind Kurs-/Lernkontext, nicht immer als Jobtitel interpretierbar."
)

add_dataset(
    "D7", "Workwise Selenium Scrape (Dynamic)", "job_ad_scrape",
    DATA_INTERIM_EXTERNAL / "scraped_workwise_dynamic_unified.parquet",
    "workwise_selenium", notes="≈609 deutsche Anzeigen; eigener Scrape; gut als DE-Gegenpol zu EN-Jobbörsen."
)

# B. BA-Skills
add_dataset(
    "D8", "BA Skillbasis (Synonym + Label)", "standard",
    DATA_INTERIM_EXTERNAL / "ba_total.parquet",
    "BA", notes="≈9.300 Kompetenzen + Synonyme; Grundlage für Terminologie-Normalisierung & Skill-DB."
)

# C. ESCO/KldB Baseline
add_dataset(
    "D9", "ESCO Skills", "standard",
    DATA_INTERIM / "esco_skills.parquet",
    "ESCO", notes="Offizielle Skill-Taxonomie laut ESCO Standard (englisch, inkl. AltLabels)."
)

add_dataset(
    "D10", "Occupation-Skill-Relations", "standard",
    DATA_INTERIM / "occupation_skill_relations.parquet",
    "ESCO", notes="Verknüpfung Beruf - Skill (essential/optional) als Basis für Soll-Profile."
)

add_dataset(
    "D11", "KldB-ESCO Mapping", "standard",
    DATA_INTERIM / "kldb_esco_mapping.parquet",
    "BA/ISCO→ESCO", notes="Offizieller Umstiegsschlüssel zur Überführung von KldB auf ESCO (über ISCO für Baseline/SOLL).."
)

add_dataset(
    "D12", "KldB-SOLL-LONG", "standard",
    DATA_PROCESSED / "kldb_skills_soll_long.parquet",
    "Derived", notes="Alle Skills pro KldB-Code in Long-Form; Grundlage Profilerweiterung."
)

add_dataset(
    "D13", "KldB-SOLL-AGG", "standard",
    DATA_PROCESSED / "kldb_skills_soll_agg.parquet",
    "Derived", notes="Aggregierte Skill-Liste pro KldB-Code (kompakt)."
)

Create and save the catalog as a DataFrame:

In [32]:
# Create a catalog
df_catalog = pd.DataFrame(catalog_rows)

# sort by ID
df_catalog["id_num"] = df_catalog["id"].str.replace("D", "", regex=False).astype(int)
df_catalog = df_catalog.sort_values("id_num").drop(columns=["id_num"]).reset_index(drop=True)

# Output
display(df_catalog)

,id,dataset_name,type,format,path,n_rows,source_name,notes
0,D1,Techsalerator Job Postings,job_ad,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,9807,kaggle_techsalerator,≈9.8k englische Jobanzeigen; zusätzliche Jobma...
1,D2,LinkedIn Job Postings (big),job_ad,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,123849,kaggle_linkedin_2023_2024_big,≈123.8k Anzeigen; große Basis für robuste Co-O...
2,D3,BA Job Ads (Jobsuche API 2025),job_ad,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,3505,ba_jobsuche_api_2025,≈3.5k deutsche Jobanzeigen; eigene/operative Q...
3,D4,Resume Dataset (Structured),cv,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,54933,kaggle_resume_structured,≈54.9k CVs/Resumes; strukturierter Resume-Date...
4,D5,LinkedIn Morocco Profiles,linkedin_profile,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,1525,kaggle_linkedin_morocco,≈1.5k Profile; Social/Profil-Quelle für Co-Occ...
5,D6,Udemy Courses,course,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,98104,kaggle_udemy_courses,"≈98.1k Kurse; Titel sind Kurs-/Lernkontext, ni..."
6,D7,Workwise Selenium Scrape (Dynamic),job_ad_scrape,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,386,workwise_selenium,≈609 deutsche Anzeigen; eigener Scrape; gut al...
7,D8,BA Skillbasis (Synonym + Label),standard,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,49324,BA,≈9.300 Kompetenzen + Synonyme; Grundlage für T...
8,D9,ESCO Skills,standard,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,13939,ESCO,Offizielle Skill-Taxonomie laut ESCO Standard ...
9,D10,Occupation-Skill-Relations,standard,parquet,C:\Users\sigle\OneDrive - Hochschule Reutlinge...,129004,ESCO,Verknüpfung Beruf - Skill (essential/optional)...


Description of the catalog columns:
- id: Unique internal identifier for each dataset (D1, D2, …). Used for referencing in the methods section
- dataset_name: Clear, readable name of the dataset
- type: Categorization by content type: job_ad, linkedin_profile, course, cv, job_ad_scrape (Workwise Selenium), standard (ESCO/KldB base data)
- format: File format of the dataset (parquet or csv)
- path: Absolute path to the file in the project. Used for traceability and reproducibility
- n_rows: Number of records (documents, rows) in the file
- source_name: Internal designation of the source (e.g., kaggle_linkedin_mid, workwise_selenium, BA, ESCO)
- notes: Additional information

In [33]:
# Save
catalog_path = DOCS_PATH / "dataset_catalog.csv"
df_catalog.to_csv(catalog_path, index=False, encoding="utf-8")

print("Datensatz-Katalog gespeichert unter:", catalog_path)

Datensatz-Katalog gespeichert unter: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\docs\dataset_catalog.csv


# Conclusion Notebook 07: documents_raw, Skill Vocabulary & Dataset Catalog

In Notebook 07, the database for the profile extension was consolidated and documented:
- The central dataset document `documents_raw.parquet` was generated from all external, already standardized text sources in `data/interim_external/` (under processed_external), ensuring the file’s stability.
- Additionally, `skills_vocab_raw.csv` was used to build an initial, controlled skill vocabulary that combines ESCO skills (EN, including alt labels) with the BA skill base (DE, including synonyms), thereby serving as a skill lexicon for the profile extension.
- Finally, `docs/dataset_catalog.csv` provides an overview of all datasets used (sources, types, sizes, usage in the pipeline).